# Reconstructed AWS-ready v5.2 Notebook

This notebook is the patched AWS-ready Jupyter version of **v5.2** for:

**LLM-Assisted RAG for Domain-Specific QA over Canadian Corporate Tax Fact Documents**

Important note: the upload contained the **v5.2 artifact bundle**, not the original source notebook. So this notebook preserves the observed v5.2 runtime stack and defaults, but some helper logic had to be rebuilt from the artifacts.

Recommended runtime:
- **EC2 GPU + JupyterLab**, or
- **SageMaker AI JupyterLab GPU space**

Core observed stack from the artifact bundle:
- embedding model: `BAAI/bge-small-en-v1.5`
- reranker: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- planner / answer / verifier model: `Qwen/Qwen2.5-1.5B-Instruct`
- 4-bit loading: enabled
- hybrid weights: dense 0.44, bm25 0.18, metadata 0.23, rerank 0.15


## 1. Environment setup

Before running the notebook on AWS, install the requirements and set `.env` from `sample.env`.

Example:
```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements_aws_v52.txt
cp sample.env .env
```


In [ ]:
from pathlib import Path
import os, json

PROJECT_DIR = Path('.').resolve()
print('Project dir:', PROJECT_DIR)
print('Files:', sorted([p.name for p in PROJECT_DIR.iterdir()]))

In [ ]:
from kpmg_tax_rag_v52_aws import Settings, KPMGTaxRAGV52AWS, preflight_summary

# Adjust these two paths if needed
BASE_DIR = os.getenv('BASE_DIR', str(PROJECT_DIR))
BUNDLE_ZIP = os.getenv('BUNDLE_ZIP_PATH', '')

settings = Settings(base_dir=Path(BASE_DIR), bundle_zip_path=Path(BUNDLE_ZIP).expanduser() if BUNDLE_ZIP else None)
rag = KPMGTaxRAGV52AWS(settings)
print(rag.reconstruction_notice)
print('Artifacts dir:', rag.settings.artifacts_dir)

## 2. Preflight check

This cell extracts the artifact bundle if needed, loads the observed run configuration, and prints a summary of the observed v5.2 bundle.

In [ ]:
rag.extract_bundle_if_needed()
rag.sync_from_s3_if_needed()
summary = preflight_summary(rag.settings.artifacts_dir)
summary

## 3. Load artifacts

This loads:
- `chunks.jsonl`
- `chunk_embeddings.npy`
- `retrieval_cache.pkl` if present

Then it builds the BM25 index in memory.

In [ ]:
rag.load_artifacts()
print('n_chunks:', len(rag.chunks))
print('embedding shape:', rag.embeddings.shape)
print('embed model:', rag.settings.embed_model_name)
print('reranker:', rag.settings.reranker_model_name)
print('llm:', rag.settings.llm_model_name)

## 4. Optional model warm-up

On GPU instances, run this to load the embedding model, reranker, and LLM into memory before asking questions.

In [ ]:
# Uncomment if you want to preload the models
# rag.load_embedder()
# rag.load_reranker()
# rag.load_llm()
# print('Models loaded.')

## 5. Ask one question

This runs the patched v5.2 flow:
1. heuristic/LLM planner
2. hybrid retrieval (dense + BM25 + metadata)
3. reranking
4. exact-mode attempt
5. LLM answer
6. LLM verifier


In [ ]:
question = 'What is the 2025 combined tax rate for a CCPC in Quebec on small business income?'
result = rag.answer_question(question)
result['full_answer']

In [ ]:
# Inspect the structured answer payload
{
    'question_type': result['question_type'],
    'baseline_answer': result['baseline_answer'],
    'deterministic_answer': result['deterministic_answer'],
    'verifier_payload': result['verifier_payload'],
    'retrieved_pages': result['retrieved_pages'],
    'top1_final_score': result['top1_final_score'],
    'exact_mode_used': result['exact_mode_used'],
}

In [ ]:
# Inspect the selected retrieval results
selected_rows = []
for row in result['retrieved']:
    chunk = row['chunk']
    selected_rows.append({
        'page': chunk.get('printed_page'),
        'type': chunk.get('content_type'),
        'section': chunk.get('section_title'),
        'row_label': chunk.get('row_label', ''),
        'final_score': row['final_score'],
        'excerpt': chunk.get('text', '')[:220],
    })
selected_rows

## 6. Evaluate the reconstructed 50-question benchmark

The file `reconstructed_eval_50q.csv` was rebuilt from the observed v5.2 output CSV so you can re-run evaluation on AWS even though the original eval input file was not part of the upload.

In [ ]:
from pathlib import Path

eval_csv = PROJECT_DIR / 'reconstructed_eval_50q.csv'
out_csv = rag.settings.outputs_dir / 'reconstructed_eval_results.csv'
out_df = rag.evaluate_csv(eval_csv, output_csv=out_csv)
summary = rag.summarize_evaluation(out_df)
summary

In [ ]:
# View the first few evaluation rows
out_df.head()

## 7. Save summary JSON

This writes a lightweight summary file into `outputs/`.

In [ ]:
summary_path = rag.settings.outputs_dir / 'reconstructed_eval_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary_path

## 8. Optional S3 upload

If `S3_BUCKET` is configured in `.env`, this uploads the local artifact directory back to S3.


In [ ]:
# Uncomment to upload artifacts to S3
# rag.sync_to_s3()

## 9. CLI equivalents

Everything in this notebook can also be run from the command line:

```bash
python kpmg_tax_rag_v52_aws.py --base-dir /home/ubuntu/e222-tax-rag-v52 --bundle-zip /path/to/kpmg_tax_rag_outputs_v52_corporate_50q.zip preflight

python kpmg_tax_rag_v52_aws.py --base-dir /home/ubuntu/e222-tax-rag-v52 --bundle-zip /path/to/kpmg_tax_rag_outputs_v52_corporate_50q.zip ask "What is the 2025 combined tax rate for a CCPC in Quebec on small business income?"

python kpmg_tax_rag_v52_aws.py --base-dir /home/ubuntu/e222-tax-rag-v52 --bundle-zip /path/to/kpmg_tax_rag_outputs_v52_corporate_50q.zip evaluate reconstructed_eval_50q.csv --output-csv outputs/re_eval.csv --summary-json outputs/re_eval_summary.json
```
